In [12]:
import os
import sys
from tqdm import tqdm
import glob
import typing
import import_ipynb

from sklearn.preprocessing import binarize,scale, robust_scale
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.svm import SVR, SVC, LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score, roc_auc_score, r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor

from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

import numpy as np
import pandas as pd
from collections import defaultdict
import librosa, opensmile

# Add current directory to path for imports
import os
current_dir = "/home2/ducvu/speech-processing-implement/codes"
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

import importlib
import config
from config import *

In [13]:
def calcualte_metrics(actual_labels:list, pred_vals:list, avg=None) -> tuple:
    if avg == None:
        f1_val, pres_val, rec_val, conf_val = score(actual_labels, pred_vals, average=None)
    else:
        f1_val, pres_val, rec_val, conf_val = score(actual_labels, pred_vals, average=avg)
    return f1_val, pres_val, rec_val, conf_val

In [14]:
def majority_voting_labels(df:pd, a_class_type:list, verbose:bool) -> tuple[int]:
    data = {
        "r_IDs": df[["r_IDs", "pred_label"]].groupby("r_IDs").mean().index.values,
        "grouped_pred_label": df[["r_IDs", "pred_label"]].groupby("r_IDs").mean().pred_label.values
    }
    df_final_test_grouped = df.merge(pd.DataFrame(data), on="r_IDs", how="inner")
    print(f"df_final_test_grouped: {df_final_test_grouped}")

    df_final_test_grouped = df_final_test_grouped.drop_duplicates(subset=["r_IDs"], keep="first")
    print(f"df_final_test_grouped_without_duplicates: {df_final_test_grouped}")

    if a_class_type == "3-way":
        threshold_val = 0.3333333
        final_pred_label_list = []
        for x in df_final_test_grouped.grouped_pred_label:
            if x < threshold_val:
                final_pred_label_list.append(0)
            elif x >= threshold_val and x < (2 * threshold_val):
                final_pred_label_list.append(1)
            else:
                final_pred_label_list.append(2)
        df_final_test_grouped["pred_label"] = final_pred_label_list
    else:
        threshold_val = 0.5
        final_pred_label_list = []
        for x in df_final_test_grouped.grouped_pred_label:
            if x < threshold_val:
                final_pred_label_list.append(0)
            else:
                final_pred_label_list.append(1)
    df_final_test_grouped.insert(len(df_final_test_grouped.columns), "final_pred_label", final_pred_label_list)


    actual_labels = df_final_test_grouped.labels

    # Calculate the initial results
    col_2_cons = "final_pred_label"
    pred_vals = df_final_test_grouped["col_2_cons"]
    f1_val, pres_val, rec_val, conf_val = calcualte_metrics(actual_labels, pred_vals, avg="macro")
    
    if verbose == 1:
        print(f"Macro F1-score: {round(f1_val, 2)}")
        print(f"Macro Precision: {round([pres_val, 2])}")
        print(f"Macro Recall: {round(rec_val, 2)}")
        print(f"Conf matrix: {conf_val}")
    
    if a_class_type == "3-way":
        precision, recall, fscore, support = score(actual_labels, pred_vals)
        if verbose == 1:
            print("Metric \t HC \t MCI \t Dementia")
            print(f"Precision \t {round(precision[0], 2)} \t {round(precision[1], 2)}, \t {round(precision[2], 2)}")
            print(f"Precision \t {round(recall[0], 2)} \t {round(recall[1], 2)}, \t {round(recall[2], 2)}")
            print(f"Precision \t {round(fscore[0], 2)} \t {round(fscore[1], 2)}, \t {round(fscore[2], 2)}")

    return f1_val, pres_val, rec_val, conf_val

In [15]:
def kfold_split(df_metadata_final:pd, 
                n_folds:int,
                label_map:dict,
                label_column:str="diagnosis",
                subject_id_column:str="participant_id",
                random_state:int=42) -> pd:

    df_metadata_copy = df_metadata_final.copy()
    # Find unique participants and their labels
    participants = df_metadata_copy[subject_id_column].unique()

    pariticpant_labels = []
    for p in participants:
        # Taking first value if duplicate
        label = df_metadata_copy[df_metadata_copy[subject_id_column] == p][label_column].iloc[0]
        pariticpant_labels.append(label)


    # Convert diagnosis to numeric
    if isinstance(pariticpant_labels[0], str):
        numeric_labels = [label_map[l] for l in pariticpant_labels]
    else:
        numeric_labels = pariticpant_labels

    df_metadata_copy["label"] = df_metadata_copy[label_column].map(label_map)
    

    # Create stratified k-fold splits
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    print(skf)

    # Add FOLD columns
    for k in range(n_folds):
        df_metadata_copy[f"FOLD_{k}"] = ''
    
    print(skf.split(participants, numeric_labels))
    # Assign train/test label for each fold
    for fold_idx ,(train_idx, test_idx) in enumerate(skf.split(participants, numeric_labels)):
        train_participants = participants[train_idx]
        test_participants = participants[test_idx]

        for p in train_participants:
            df_metadata_copy.loc[df_metadata_copy[subject_id_column] == p, f"FOLD_{fold_idx}"] = "TRAIN"
        for p in test_participants:
            df_metadata_copy.loc[df_metadata_copy[subject_id_column] == p, f"FOLD_{fold_idx}"] = "TEST"
    return df_metadata_copy

In [16]:
def train_classifier(df_train:pd,
                    feat_names:list[str],
                    classifier_type:str,
                    class_type_chosen:str,
                    n_jobs:int):
    # Prepare data
    x_train = np.array(df_train[feat_names])
    y_train = np.array(df_train["label"].values)

    # Scale feature
    x_train_scaled = robust_scale(x_train)
    if class_type_chosen == "grid":
        pass
    elif class_type_chosen == "simple":
       # Use default parameters
        if classifier_type == 'LR':
            model = LogisticRegression(max_iter=int(1e+20), n_jobs=n_jobs)
        elif classifier_type == 'SVM':
            model = SVC(probability=True)
        elif classifier_type == 'MLP':
            model = MLPClassifier(random_state=1, max_iter=int(1e+20))
        else:
            raise ValueError(f"Unknown classifier type: {classifier_type}")
        
        trained_model = model.fit(x_train_scaled, y_train)
    
    else:
        raise ValueError(f"Unknown optimization type: {class_type_chosen}")
    return trained_model 
    

def evaluate_fold(trained_model, 
                    df_test:pd, 
                    feat_names:list[str], 
                    way_classification:list[str], 
                    majority_voting_labels:tuple) -> tuple[int]:
    pass

def run_single_experiment(classifier_chosen:list[int],
                            task_chosen:list[str],
                            class_type_chosen:list[str],
                            opensmile_feature:list[str],
                            N_FOLDS:int,
                            CV_SCORER:str,
                            N_JOBS:int,
                            majority_voting_labels:tuple) -> defaultdict:
    pass

In [17]:
# Hyperparameter
CV_SCORER = config.CV_SCORER
N_FOLDS = config.N_FOLDS

# Directory
DATA_PATH = config.DATA_PATH
FEATS_PATH = config.FEATS_PATH
RESULTS_PATH = config.RESULTS_PATH

# List acoustic feature
LIST_ACOUSTIC = config.LIST_ACOUSTIC

# List classifier and chosen
LIST_CLASSIFIER_NAME = config.LIST_CLASSIFIER_NAME
CLASSIFIER_CHOSEN = config.CLASSIFIER_CHOSEN

# List task chosen
TASK_CHOSEN = config.TASK_CHOSEN

# List class type chosen
CLASS_TYPE_CHOSEN = config.CLASS_TYPE_CHOSEN

# Way for classifying
WAY_CLASSIFICATION = config.WAY_CLASSIFICATION

# Mapping label
LABEL_MAP = config.LABEL_MAP

In [ ]:
df_metadata_final = pd.read_csv(f"{DATA_PATH}/metadata.csv")
for i in range(N_FOLDS):
    df_metadata_final = df_metadata_final.drop(columns=f"FOLD_{i}")

df_metadata_final = df_metadata_final.drop(columns=f"labels")
df_metadata_final

In [ ]:
df_metadata_final = kfold_split(df_metadata_final, 5, LABEL_MAP, "diagnosis", "participant_id", 42)
df_metadata_final